In [1]:
from pathlib import Path

import sqlite3
import pandas as pd

from src.agent import Agent
from src.tools import _load_pdf_text
from src.config import settings

agent = Agent()

path = Path("data")

Você é o assistente de uma loja brasileira chamada Empório da Música.
Responda sempre em português do Brasil, de forma clara e objetiva e amigável, seguindo as informações e a persona contida no documento da loja.

Você tem uma ferramenta:
- consult_sql: consulta o banco de dados da loja (pedidos, clientes, produtos,
   categorias, itens de pedido). Use para qualquer pergunta que dependa de DADOS.
 
Regras:
- Se a pergunta é sobre dados do banco, responda somente com base no que a ferramenta retornar. Nunca invente.
- Se a ferramenta não trouxer a informação, diga que você não sabe responder à pergunta feita.
- Valores monetários estão em reais (BRL).
- Para consultas SQL, use apenas as tabelas e colunas do esquema informado.
- Para busca por nome de produto, prefira LIKE '%termo%' (sem diferenciar maiúsculas).
- Nunca exponha dados pessoais sensíveis de clientes (e-mail, telefone, documentos).
 
Esquema do banco de dados (SQLite):
 
CREATE TABLE "categories" (
"category_id" INTEGER,
 

In [2]:
question = "qual o endereço da loja?"
response = agent.ask(question)

In [3]:
response

{'messages': [HumanMessage(content='qual o endereço da loja?', additional_kwargs={}, response_metadata={}, id='1189cf38-811e-4b2a-9a86-54481c278e53'),
  AIMessage(content='O endereço da loja é: **Rua 14 de Maio, 3200 — Centro, Campo Grande - MS, CEP 79202-333**. 😊', additional_kwargs={}, response_metadata={'model': 'qwen3:8b', 'created_at': '2026-11-04T12:14:36.474748Z', 'done': True, 'done_reason': 'stop', 'total_duration': 44631693875, 'load_duration': 2580167, 'prompt_eval_count': 5057, 'prompt_eval_duration': 33592982000, 'eval_count': 178, 'eval_duration': 10820065000, 'logprobs': None, 'model_name': 'qwen3:8b', 'model_provider': 'ollama'}, id='lc_run--01a1a67a-db5f-759f-80a0-4028f9d82b10-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 5057, 'output_tokens': 178, 'total_tokens': 5235})]}

In [ ]:
csv_files = list(path.glob("*.csv"))
dfs = [pd.read_csv(file) for file in csv_files]

In [ ]:
dfs[5].head()

In [ ]:
conn = sqlite3.connect("data/database.db")

for f in csv_files:
    df = pd.read_csv(f)

    table_name = f.stem.split(" - ", maxsplit=1)[1]
    
    df.to_sql(
        name=table_name,
        con=conn,
        if_exists="replace",
        index=False
    )

conn.close()

In [ ]:
def query_db(query: str) -> pd.DataFrame:
    with sqlite3.connect("data/database.db") as conn:
        return pd.read_sql_query(query, conn)

In [ ]:
query = """
SELECT
    p.product_id,
    p.name AS product_name,
    c.name AS category,
    SUM(oi.quantity) AS units_sold,
    SUM(oi.quantity * p.price_brl) AS estimated_revenue
FROM order_items oi
JOIN products p
    ON oi.product_id = p.product_id
JOIN categories c
    ON p.category_id = c.category_id
JOIN orders o
    ON oi.order_id = o.order_id
WHERE o.status != 'cancelled'
GROUP BY p.product_id, p.name, c.name
ORDER BY units_sold DESC;
"""

df = query_db(query)

In [ ]:
df